# 🦠 COVID-19 Twitter Discourse Analysis
## Sentiment Analysis & Topic Modelling on WHO-Related Tweets (2020)

---

**Author:** Chukwuemeka I. Ogbonnaya  
**Original Thesis:** MSc in Microdata Analysis — Dalarna University, Sweden (2022)  
**Modernised:** 2026 — Refactored with current best practices in NLP and data science  

---

## 📋 Project Overview

This notebook analyses **95,342 English-language tweets** related to the WHO's COVID-19 communications during the early pandemic period (March–June 2020). The analysis applies two core NLP techniques:

1. **Sentiment Analysis** — classifying tweet polarity (positive / negative / neutral) using TextBlob and VADER
2. **Topic Modelling** — discovering latent themes in the tweet corpus using Latent Dirichlet Allocation (LDA)

The goal is to understand **what people were saying** and **how they felt** about WHO-related COVID-19 information on Twitter during the pandemic's early phase.

---

## 🗂️ Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Data Loading & Initial Inspection](#2-data-loading)
3. [Exploratory Data Analysis (EDA)](#3-eda)
4. [Text Preprocessing Pipeline](#4-text-preprocessing)
5. [Sentiment Analysis](#5-sentiment-analysis)
6. [Topic Modelling with LDA](#6-topic-modelling)
7. [Topic Coherence Optimisation](#7-coherence)
8. [Temporal Analysis](#8-temporal-analysis)
9. [Key Findings & Conclusions](#9-conclusions)

---

### 📦 Dependencies
```
pip install pandas numpy matplotlib seaborn plotly wordcloud
pip install nltk textblob vaderSentiment gensim pyLDAvis
pip install scikit-learn tqdm
```

## 1. Environment Setup

All imports are consolidated in a single cell with clear grouping and no duplicates.
We download only the NLTK resources we actually use.

In [ ]:
# ── Standard library ──────────────────────────────────────────────
import re
import os
import warnings
from pathlib import Path
from collections import Counter

# ── Data manipulation ─────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud

# ── NLP — NLTK ────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ── NLP — Sentiment ───────────────────────────────────────────────
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ── NLP — Topic Modelling ─────────────────────────────────────────
import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel, CoherenceModel
from gensim.utils import simple_preprocess
import pyLDAvis
import pyLDAvis.gensim_models  # updated import for gensim 4.x

# ── Progress tracking ─────────────────────────────────────────────
from tqdm import tqdm
tqdm.pandas()  # enables df.progress_apply()

# ── Settings ──────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

# Chart style
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
PALETTE = ['#1a3558', '#c8963e', '#2e7d32', '#c62828',
           '#1565c0', '#6a1b9a', '#00838f', '#558b2f']
sns.set_palette(PALETTE)

# Create output folder
Path('outputs').mkdir(exist_ok=True)
Path('outputs/charts').mkdir(exist_ok=True)

# Download required NLTK data
for resource in ['stopwords', 'wordnet', 'punkt', 'vader_lexicon']:
    nltk.download(resource, quiet=True)

print('✅ Environment ready')
print(f'   pandas  : {pd.__version__}')
print(f'   gensim  : {gensim.__version__}')
print(f'   numpy   : {np.__version__}')

## 2. Data Loading & Initial Inspection

The dataset contains English-language tweets mentioning WHO and COVID-19,
collected between March and June 2020.

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────
# Update this path to point to your local copy of the dataset
DATA_PATH = Path('data/english_tweets.csv')

df = pd.read_csv(DATA_PATH)

# Parse date column with explicit UTC format
df['created_at'] = pd.to_datetime(
    df['created_at'],
    format='%a %b %d %H:%M:%S +0000 %Y',
    errors='coerce',
    utc=True
)

print(f'✅ Dataset loaded')
print(f'   Shape       : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Date range  : {df["created_at"].min().date()} to {df["created_at"].max().date()}')
print(f'   Missing text: {df["text"].isna().sum():,}')
df.head()

In [ ]:
# ── Basic data quality check ──────────────────────────────────────
print('=== DATA TYPES ===')
print(df.dtypes)

print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values ✅')

print(f'\n=== DUPLICATE ROWS ===')
dupes = df.duplicated(subset='text').sum()
print(f'{dupes:,} duplicate tweets found')

print(f'\n=== LANGUAGES ===')
if 'lang' in df.columns:
    print(df['lang'].value_counts().head())

In [ ]:
# ── Initial data cleaning ─────────────────────────────────────────
rows_before = len(df)

# Remove duplicates
df = df.drop_duplicates(subset='text').reset_index(drop=True)

# Drop rows with missing text or dates
df = df.dropna(subset=['text', 'created_at']).reset_index(drop=True)

# Keep only English tweets if language column exists
if 'lang' in df.columns:
    df = df[df['lang'] == 'en'].reset_index(drop=True)

# Extract temporal features
df['year']        = df['created_at'].dt.year
df['month']       = df['created_at'].dt.month
df['month_name']  = df['created_at'].dt.month_name()
df['week']        = df['created_at'].dt.isocalendar().week.astype(int)
df['day_of_week'] = df['created_at'].dt.day_name()
df['hour']        = df['created_at'].dt.hour
df['date']        = df['created_at'].dt.date

print(f'Rows before cleaning : {rows_before:,}')
print(f'Rows after cleaning  : {len(df):,}')
print(f'Rows removed         : {rows_before - len(df):,}')
print('\n✅ Data cleaning complete')

## 3. Exploratory Data Analysis (EDA)

Before modelling, we explore the temporal distribution of tweets,
tweet volume patterns, and basic text characteristics.

In [ ]:
# ── Tweet volume over time ────────────────────────────────────────
daily_counts = df.groupby('date').size().reset_index(name='tweet_count')
daily_counts['date'] = pd.to_datetime(daily_counts['date'])
daily_counts['rolling_7d'] = daily_counts['tweet_count'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.fill_between(daily_counts['date'], daily_counts['tweet_count'],
                alpha=0.2, color=PALETTE[0], label='Daily count')
ax.plot(daily_counts['date'], daily_counts['tweet_count'],
        color=PALETTE[0], linewidth=1, alpha=0.6)
ax.plot(daily_counts['date'], daily_counts['rolling_7d'],
        color=PALETTE[1], linewidth=2.5, label='7-day rolling average')

ax.set_title('Daily Tweet Volume — WHO COVID-19 Discourse (Mar–Jun 2020)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Tweets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.savefig('outputs/charts/01_daily_tweet_volume.png', dpi=150)
plt.show()
print('✅ Chart saved')

In [ ]:
# ── Posting patterns: by hour and day of week ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By hour
hourly = df['hour'].value_counts().sort_index()
axes[0].bar(hourly.index, hourly.values,
            color=PALETTE[0], alpha=0.85, edgecolor='white')
axes[0].set_title('Tweet Volume by Hour of Day (UTC)', fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Tweet Count')
axes[0].set_xticks(range(0, 24))

# By day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df['day_of_week'].value_counts().reindex(day_order)
axes[1].bar(dow.index, dow.values,
            color=PALETTE[1], alpha=0.85, edgecolor='white')
axes[1].set_title('Tweet Volume by Day of Week', fontweight='bold')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Tweet Count')
axes[1].set_xticklabels(day_order, rotation=30, ha='right')

plt.suptitle('Temporal Posting Patterns', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/02_posting_patterns.png', dpi=150)
plt.show()

In [ ]:
# ── Text length distribution ──────────────────────────────────────
df['tweet_length']     = df['text'].str.len()
df['word_count']       = df['text'].str.split().str.len()
df['hashtag_count']    = df['text'].str.count(r'#\w+')
df['mention_count']    = df['text'].str.count(r'@\w+')
df['url_count']        = df['text'].str.count(r'https?://\S+')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title, color in [
    (axes[0], 'tweet_length', 'Tweet Character Length', PALETTE[0]),
    (axes[1], 'word_count',   'Word Count per Tweet',   PALETTE[1]),
    (axes[2], 'hashtag_count','Hashtags per Tweet',     PALETTE[2]),
]:
    ax.hist(df[col], bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(df[col].median(), color='red', linestyle='--',
               linewidth=1.5, label=f'Median: {df[col].median():.0f}')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Tweet Text Characteristics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/03_text_characteristics.png', dpi=150)
plt.show()

print('Text Statistics Summary:')
print(df[['tweet_length','word_count','hashtag_count','mention_count']].describe().round(2))

## 4. Text Preprocessing Pipeline

A clean, modular preprocessing function that handles all Twitter-specific
noise: URLs, mentions, hashtags, emojis, and special characters.
Each step is documented and testable.

In [ ]:
# ── Text preprocessing pipeline ───────────────────────────────────
STOP_WORDS = set(stopwords.words('english'))

# Add Twitter-specific and COVID-specific stopwords
CUSTOM_STOPWORDS = {
    'rt', 'via', 'amp', 'covid', 'covid19', 'coronavirus',
    'who', 'pandemic', 'virus', 'http', 'https', 'co',
    'would', 'could', 'get', 'also', 'us', 'said', 'say',
    'one', 'like', 'new', 'know'
}
STOP_WORDS.update(CUSTOM_STOPWORDS)

lemmatizer = WordNetLemmatizer()


def clean_tweet(text: str) -> str:
    """Clean raw tweet text for sentiment analysis.
    Removes noise while preserving semantic content.

    Args:
        text: Raw tweet string

    Returns:
        Cleaned lowercase string
    """
    text = str(text).lower()
    text = re.sub(r'https?://\S+', '', text)       # Remove URLs
    text = re.sub(r'@\w+', '', text)               # Remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)          # Keep hashtag words, remove #
    text = re.sub(r'[^\x00-\x7F]+', '', text)     # Remove non-ASCII (emojis)
    text = re.sub(r'\d+', '', text)               # Remove numbers
    text = re.sub(r'[^\w\s]', ' ', text)          # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()      # Collapse whitespace
    return text


def tokenize_and_lemmatize(text: str) -> list[str]:
    """Tokenize, remove stopwords, and lemmatize cleaned text.

    Args:
        text: Pre-cleaned tweet string

    Returns:
        List of lemmatized tokens (min length 3 chars)
    """
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in STOP_WORDS and len(token) >= 3
    ]
    return tokens


# Apply preprocessing with progress bar
print('Cleaning tweet text...')
df['text_clean'] = df['text'].progress_apply(clean_tweet)

print('Tokenising and lemmatising...')
df['tokens'] = df['text_clean'].progress_apply(tokenize_and_lemmatize)
df['token_count'] = df['tokens'].str.len()

# Remove tweets with fewer than 3 tokens after cleaning
df = df[df['token_count'] >= 3].reset_index(drop=True)

print(f'\n✅ Preprocessing complete')
print(f'   Remaining tweets : {len(df):,}')
print(f'\nSample cleaned output:')
for i in range(3):
    print(f'\n  Original : {df["text"].iloc[i][:100]}')
    print(f'  Cleaned  : {df["text_clean"].iloc[i][:100]}')
    print(f'  Tokens   : {df["tokens"].iloc[i][:10]}')

In [ ]:
# ── Word frequency and word cloud ─────────────────────────────────
all_tokens = [token for tokens in df['tokens'] for token in tokens]
token_freq = Counter(all_tokens)
top_50 = token_freq.most_common(50)

# Word cloud
wc = WordCloud(
    width=1400, height=600,
    background_color='white',
    colormap='Blues',
    max_words=100,
    max_font_size=120,
    random_state=42
).generate_from_frequencies(dict(token_freq))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Word cloud
axes[0].imshow(wc, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Most Frequent Terms in COVID-19 Tweets',
                  fontweight='bold', fontsize=13)

# Top 20 bar chart
top20_words, top20_counts = zip(*top_50[:20])
axes[1].barh(list(top20_words)[::-1], list(top20_counts)[::-1],
             color=PALETTE[0], alpha=0.85)
axes[1].set_title('Top 20 Most Frequent Terms', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Frequency')

plt.suptitle('Token Frequency Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/04_word_frequency.png', dpi=150)
plt.show()

## 5. Sentiment Analysis

We use two complementary approaches:
- **TextBlob**: Polarity (-1 to +1) and Subjectivity (0 to 1)
- **VADER**: Specifically tuned for social media text, handles slang, caps, punctuation

VADER is generally more accurate for Twitter data — we use it as the primary classifier.

In [ ]:
# ── VADER sentiment scoring ───────────────────────────────────────
vader = SentimentIntensityAnalyzer()


def get_vader_scores(text: str) -> dict:
    """Return VADER compound, positive, negative, neutral scores."""
    return vader.polarity_scores(str(text))


def classify_sentiment(compound: float) -> str:
    """Classify sentiment based on VADER compound score.

    Thresholds follow VADER documentation:
    compound >= 0.05  → Positive
    compound <= -0.05 → Negative
    else              → Neutral
    """
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    return 'Neutral'


print('Running VADER sentiment analysis...')
vader_scores = df['text'].progress_apply(get_vader_scores)
df['vader_compound']  = vader_scores.apply(lambda x: x['compound'])
df['vader_positive']  = vader_scores.apply(lambda x: x['pos'])
df['vader_negative']  = vader_scores.apply(lambda x: x['neg'])
df['vader_neutral']   = vader_scores.apply(lambda x: x['neu'])
df['sentiment_vader'] = df['vader_compound'].apply(classify_sentiment)

# ── TextBlob sentiment ────────────────────────────────────────────
print('Running TextBlob sentiment analysis...')
df['textblob_polarity']     = df['text_clean'].progress_apply(
    lambda x: TextBlob(x).sentiment.polarity
)
df['textblob_subjectivity'] = df['text_clean'].progress_apply(
    lambda x: TextBlob(x).sentiment.subjectivity
)
df['sentiment_textblob'] = df['textblob_polarity'].apply(classify_sentiment)

print('\n✅ Sentiment analysis complete')
print('\nVADER Sentiment Distribution:')
print(df['sentiment_vader'].value_counts())
print(f'\nPercentages:')
print(df['sentiment_vader'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
# ── Sentiment visualisation ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. VADER sentiment distribution (pie)
sentiment_counts = df['sentiment_vader'].value_counts()
colors_sent = {'Positive': '#2e7d32', 'Neutral': '#1565c0', 'Negative': '#c62828'}
axes[0,0].pie(
    sentiment_counts.values,
    labels=sentiment_counts.index,
    autopct='%1.1f%%',
    colors=[colors_sent[k] for k in sentiment_counts.index],
    startangle=90,
    pctdistance=0.85
)
axes[0,0].set_title('VADER Sentiment Distribution', fontweight='bold')

# 2. VADER compound score distribution
axes[0,1].hist(df['vader_compound'], bins=60,
               color=PALETTE[0], alpha=0.85, edgecolor='white')
axes[0,1].axvline(0.05,  color='green',  linestyle='--', linewidth=1.5, label='Positive threshold')
axes[0,1].axvline(-0.05, color='red',    linestyle='--', linewidth=1.5, label='Negative threshold')
axes[0,1].axvline(df['vader_compound'].mean(), color='orange',
                  linewidth=2, label=f'Mean: {df["vader_compound"].mean():.3f}')
axes[0,1].set_title('VADER Compound Score Distribution', fontweight='bold')
axes[0,1].set_xlabel('Compound Score')
axes[0,1].legend(fontsize=9)

# 3. TextBlob polarity vs subjectivity scatter
sample = df.sample(min(3000, len(df)), random_state=42)
scatter_colors = [colors_sent[s] for s in sample['sentiment_vader']]
axes[1,0].scatter(sample['textblob_polarity'], sample['textblob_subjectivity'],
                  c=scatter_colors, alpha=0.3, s=8)
axes[1,0].set_title('TextBlob: Polarity vs Subjectivity', fontweight='bold')
axes[1,0].set_xlabel('Polarity (Negative ← → Positive)')
axes[1,0].set_ylabel('Subjectivity (Objective ← → Subjective)')
axes[1,0].axvline(0, color='gray', linewidth=0.8)

# 4. TextBlob vs VADER agreement
agreement = (df['sentiment_vader'] == df['sentiment_textblob']).mean() * 100
ct = pd.crosstab(df['sentiment_vader'], df['sentiment_textblob'], normalize='index') * 100
sns.heatmap(ct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1,1],
            cbar_kws={'label': '% of VADER class'})
axes[1,1].set_title(f'VADER vs TextBlob Agreement\n(Overall: {agreement:.1f}%)',
                    fontweight='bold')
axes[1,1].set_xlabel('TextBlob Label')
axes[1,1].set_ylabel('VADER Label')

plt.suptitle('Sentiment Analysis Results', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/05_sentiment_analysis.png', dpi=150)
plt.show()

print(f'\nVADER vs TextBlob agreement rate: {agreement:.1f}%')

In [ ]:
# ── Sentiment over time ───────────────────────────────────────────
weekly_sentiment = df.groupby(['week', 'sentiment_vader']).size().unstack(fill_value=0)
weekly_sentiment_pct = weekly_sentiment.div(weekly_sentiment.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Absolute counts
weekly_sentiment.plot(kind='bar', ax=axes[0], stacked=True,
    color=[colors_sent.get(c, '#999') for c in weekly_sentiment.columns],
    alpha=0.85, edgecolor='white', width=0.8)
axes[0].set_title('Weekly Tweet Volume by Sentiment', fontweight='bold')
axes[0].set_xlabel('Week Number')
axes[0].set_ylabel('Tweet Count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Sentiment')

# Percentage
weekly_sentiment_pct.plot(kind='area', ax=axes[1],
    color=[colors_sent.get(c, '#999') for c in weekly_sentiment_pct.columns],
    alpha=0.7)
axes[1].set_title('Weekly Sentiment Share (%)', fontweight='bold')
axes[1].set_xlabel('Week Number')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Sentiment')

plt.suptitle('Sentiment Trends Over Time', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/06_sentiment_over_time.png', dpi=150)
plt.show()

## 6. Topic Modelling with LDA

Latent Dirichlet Allocation (LDA) is applied to discover latent topics
across the tweet corpus. We use Gensim's updated 4.x API.

**Key design decisions:**
- Bigrams are added to capture multi-word phrases (e.g. 'public_health')
- Rare (<15 docs) and very common (>50% of docs) terms are filtered
- We train models for k=3 to k=12 topics and select by coherence score

In [ ]:
# ── Prepare corpus for LDA ────────────────────────────────────────
texts = df['tokens'].tolist()

# Build bigram model to capture common phrases
bigram_phrases = gensim.models.Phrases(
    texts,
    min_count=20,
    threshold=50
)
bigram_mod = gensim.models.phrases.Phraser(bigram_phrases)
texts_bigrams = [bigram_mod[text] for text in texts]

# Build dictionary and filter extremes
dictionary = corpora.Dictionary(texts_bigrams)
dictionary.filter_extremes(
    no_below=15,   # word must appear in at least 15 documents
    no_above=0.5   # word must not appear in more than 50% of documents
)

# Build document-term matrix (bag-of-words corpus)
bow_corpus = [dictionary.doc2bow(text) for text in texts_bigrams]

print(f'✅ Corpus ready for LDA')
print(f'   Documents  : {len(bow_corpus):,}')
print(f'   Unique terms in dictionary: {len(dictionary):,}')

# Store corpus alongside dataframe
df['bow'] = bow_corpus

## 7. Topic Coherence Optimisation

We evaluate models with k=3 to k=12 topics using the c_v coherence metric.
Higher coherence indicates more interpretable, meaningful topics.

In [ ]:
# ── Train LDA models across a range of topic counts ──────────────
def train_lda_model(corpus, dictionary, num_topics: int, seed: int = 42) -> LdaModel:
    """Train an LDA model with fixed random seed for reproducibility."""
    return LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=seed,
        passes=10,
        alpha='auto',        # learn asymmetric topic-document distribution
        eta='auto',          # learn asymmetric word-topic distribution
        per_word_topics=True
    )


def compute_coherence(model, texts, dictionary) -> float:
    """Compute c_v coherence score for a trained LDA model."""
    cm = CoherenceModel(
        model=model,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()


# Train models for k = 3 to 12
TOPIC_RANGE = range(3, 13)
results = []

print('Training LDA models and evaluating coherence...')
for k in tqdm(TOPIC_RANGE, desc='Topic counts'):
    model = train_lda_model(bow_corpus, dictionary, k)
    coherence = compute_coherence(model, texts_bigrams, dictionary)
    results.append({'num_topics': k, 'coherence': coherence, 'model': model})
    print(f'   k={k:2d} | Coherence: {coherence:.4f}')

results_df = pd.DataFrame(results)[['num_topics', 'coherence']]

# Select best model
best_idx    = results_df['coherence'].idxmax()
best_k      = results_df.loc[best_idx, 'num_topics']
best_score  = results_df.loc[best_idx, 'coherence']
best_model  = results[best_idx]['model']

print(f'\n✅ Best model: k={best_k} topics (coherence={best_score:.4f})')

In [ ]:
# ── Coherence score plot ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_df['num_topics'], results_df['coherence'],
        marker='o', linewidth=2.5, color=PALETTE[0], markersize=8)
ax.axvline(best_k, color=PALETTE[1], linestyle='--', linewidth=2,
           label=f'Best k={best_k} (coherence={best_score:.4f})')
ax.scatter([best_k], [best_score], color=PALETTE[1], s=120, zorder=5)
ax.set_title('LDA Coherence Score by Number of Topics',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Topics (k)')
ax.set_ylabel('Coherence Score (c_v)')
ax.set_xticks(list(TOPIC_RANGE))
ax.legend()
plt.tight_layout()
plt.savefig('outputs/charts/07_coherence_scores.png', dpi=150)
plt.show()

In [ ]:
# ── Display discovered topics ─────────────────────────────────────
print(f'=== TOP KEYWORDS PER TOPIC (k={best_k}) ===\n')
topic_labels = {}  # fill these in after inspecting the output

for topic_id in range(best_k):
    top_words = best_model.show_topic(topic_id, topn=12)
    keywords  = ', '.join([w for w, _ in top_words])
    weights   = [round(w, 4) for _, w in top_words]
    print(f'Topic {topic_id+1:02d}: {keywords}')
    print(f'  Weights: {weights}\n')

print('\n💡 Inspect the keywords above and add human-readable labels below:')
print('   e.g. topic_labels = {0: "Public Health Measures", 1: "Economic Impact", ...}')

In [ ]:
# ── Assign dominant topic to each tweet ───────────────────────────
def get_dominant_topic(lda_model, bow: list) -> tuple:
    """Return the dominant topic id and its contribution score."""
    topic_dist = lda_model.get_document_topics(bow)
    if not topic_dist:
        return np.nan, np.nan
    dominant = max(topic_dist, key=lambda x: x[1])
    return dominant[0], round(dominant[1], 4)


print('Assigning dominant topics to tweets...')
topic_results = df['bow'].progress_apply(
    lambda bow: get_dominant_topic(best_model, bow)
)
df['dominant_topic']  = topic_results.apply(lambda x: x[0])
df['topic_score']     = topic_results.apply(lambda x: x[1])

# Add human-readable topic keywords
topic_keywords_map = {
    i: ', '.join([w for w, _ in best_model.show_topic(i, topn=5)])
    for i in range(best_k)
}
df['topic_keywords'] = df['dominant_topic'].map(topic_keywords_map)

print('\n✅ Topic assignment complete')
print('\nTopic Distribution:')
print(df['dominant_topic'].value_counts().sort_index())

In [ ]:
# ── Interactive pyLDAvis visualisation ────────────────────────────
# Note: uses updated gensim_models API for gensim 4.x
pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim_models.prepare(
    best_model,
    bow_corpus,
    dictionary,
    sort_topics=False
)

# Save for GitHub / sharing
pyLDAvis.save_html(vis, 'outputs/lda_visualisation.html')
print('✅ Interactive LDA visualisation saved to outputs/lda_visualisation.html')

vis  # Display inline

## 8. Temporal Analysis

How do sentiment and topic distribution evolve over the pandemic period?

In [ ]:
# ── Weekly average sentiment score ───────────────────────────────
weekly_avg = df.groupby('week').agg(
    avg_compound=('vader_compound', 'mean'),
    tweet_volume=('vader_compound', 'count'),
    avg_subjectivity=('textblob_subjectivity', 'mean')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Compound score over time
axes[0].fill_between(weekly_avg['week'],
                     weekly_avg['avg_compound'],
                     0, alpha=0.3,
                     color=weekly_avg['avg_compound'].apply(
                         lambda x: '#2e7d32' if x >= 0 else '#c62828'))
axes[0].plot(weekly_avg['week'], weekly_avg['avg_compound'],
             color=PALETTE[0], linewidth=2.5, marker='o', markersize=5)
axes[0].axhline(0, color='gray', linewidth=1, linestyle='--')
axes[0].axhline(0.05,  color='green', linewidth=0.8, linestyle=':', alpha=0.7)
axes[0].axhline(-0.05, color='red',   linewidth=0.8, linestyle=':', alpha=0.7)
axes[0].set_title('Weekly Average VADER Compound Sentiment Score',
                  fontweight='bold')
axes[0].set_xlabel('Week Number')
axes[0].set_ylabel('Mean Compound Score')
axes[0].set_xticks(weekly_avg['week'])

# Subjectivity over time
axes[1].plot(weekly_avg['week'], weekly_avg['avg_subjectivity'],
             color=PALETTE[1], linewidth=2.5, marker='s', markersize=5)
axes[1].fill_between(weekly_avg['week'], weekly_avg['avg_subjectivity'],
                     alpha=0.2, color=PALETTE[1])
axes[1].set_title('Weekly Average TextBlob Subjectivity Score',
                  fontweight='bold')
axes[1].set_xlabel('Week Number')
axes[1].set_ylabel('Mean Subjectivity (0=Objective, 1=Subjective)')
axes[1].set_xticks(weekly_avg['week'])

plt.suptitle('Sentiment Evolution Over Time', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/08_temporal_sentiment.png', dpi=150)
plt.show()

In [ ]:
# ── Sentiment distribution per topic ─────────────────────────────
topic_sent = df.groupby(['dominant_topic', 'sentiment_vader']).size().unstack(fill_value=0)
topic_sent_pct = topic_sent.div(topic_sent.sum(axis=1), axis=0) * 100
topic_sent_pct.index = [f'Topic {i+1}' for i in topic_sent_pct.index]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Stacked bar
topic_sent_pct.plot(kind='bar', ax=axes[0], stacked=True,
    color=[colors_sent.get(c, '#999') for c in topic_sent_pct.columns],
    alpha=0.85, edgecolor='white', width=0.7)
axes[0].set_title('Sentiment Distribution per Topic (%)', fontweight='bold')
axes[0].set_xlabel('Topic')
axes[0].set_ylabel('Percentage')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Sentiment', loc='upper right')

# Average compound score per topic
avg_compound_by_topic = df.groupby('dominant_topic')['vader_compound'].mean()
bar_colors = ['#2e7d32' if v >= 0 else '#c62828' for v in avg_compound_by_topic.values]
axes[1].bar([f'Topic {i+1}' for i in avg_compound_by_topic.index],
            avg_compound_by_topic.values,
            color=bar_colors, alpha=0.85)
axes[1].axhline(0, color='gray', linewidth=1)
axes[1].set_title('Average Sentiment Score per Topic\n(Green=Positive, Red=Negative)',
                  fontweight='bold')
axes[1].set_xlabel('Topic')
axes[1].set_ylabel('Mean VADER Compound Score')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Topic × Sentiment Cross-Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/09_topic_sentiment_cross.png', dpi=150)
plt.show()

## 9. Key Findings & Conclusions

A structured summary of the analysis results for thesis documentation and reporting.

In [ ]:
# ── Auto-generated findings summary ──────────────────────────────
sentiment_pct = df['sentiment_vader'].value_counts(normalize=True) * 100
dominant_sentiment = sentiment_pct.idxmax()
avg_compound = df['vader_compound'].mean()
avg_subjectivity = df['textblob_subjectivity'].mean()
peak_week = weekly_avg.loc[weekly_avg['tweet_volume'].idxmax(), 'week']
most_common_topic = df['dominant_topic'].value_counts().idxmax()

print('=' * 65)
print('       📋 KEY FINDINGS SUMMARY')
print('=' * 65)

print(f'\n📊 DATASET')
print(f'   Total tweets analysed : {len(df):,}')
print(f'   Date range            : Mar – Jun 2020')
print(f'   Peak activity week    : Week {peak_week}')

print(f'\n💬 SENTIMENT (VADER)')
for sent, pct in sentiment_pct.items():
    print(f'   {sent:<10}: {pct:.1f}%')
print(f'   Mean compound score   : {avg_compound:.4f}')
print(f'   Mean subjectivity     : {avg_subjectivity:.4f}')
print(f'   Dominant sentiment    : {dominant_sentiment}')

print(f'\n🔍 TOPIC MODELLING')
print(f'   Optimal number of topics : {best_k}')
print(f'   Best coherence score     : {best_score:.4f}')
print(f'   Most dominant topic      : Topic {most_common_topic + 1}')
print(f'     Keywords: {topic_keywords_map[most_common_topic]}')

print(f'\n💡 INTERPRETATION')
if avg_compound > 0:
    print(f'   The overall discourse was mildly positive (mean={avg_compound:.3f}),')
    print(f'   suggesting public trust in WHO communications during the early pandemic.')
else:
    print(f'   The overall discourse was mildly negative (mean={avg_compound:.3f}),')
    print(f'   suggesting public concern or criticism of WHO communications.')
print(f'   Subjectivity ({avg_subjectivity:.3f}) suggests a mix of factual')
print(f'   reporting and personal opinion in the tweet corpus.')
print('=' * 65)

In [ ]:
# ── Save processed results ────────────────────────────────────────
output_cols = [
    'text', 'text_clean', 'created_at', 'date', 'week', 'hour',
    'tweet_length', 'word_count', 'hashtag_count',
    'vader_compound', 'vader_positive', 'vader_negative',
    'textblob_polarity', 'textblob_subjectivity',
    'sentiment_vader', 'sentiment_textblob',
    'dominant_topic', 'topic_score', 'topic_keywords'
]

results_path = Path('outputs/tweets_analysed.csv')
df[output_cols].to_csv(results_path, index=False)

print(f'✅ Results saved to {results_path}')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {len(output_cols)}')
print(f'\nOutputs directory:')
for f in sorted(Path('outputs').rglob('*')):
    if f.is_file():
        print(f'   {f}')

---

## 📁 Project Structure

```
covid19-twitter-nlp-analysis/
│
├── data/
│   └── english_tweets.csv          ← Raw dataset (95,342 tweets)
│
├── outputs/
│   ├── charts/                     ← All saved visualisations
│   ├── tweets_analysed.csv         ← Processed results with sentiment + topics
│   └── lda_visualisation.html      ← Interactive pyLDAvis output
│
├── COVID19_Twitter_NLP_Analysis.ipynb  ← This notebook
└── README.md
```

---

## 🔬 Technical Stack

| Layer | Tool | Version |
|---|---|---|
| Data manipulation | pandas, numpy | 2.x, 1.x |
| Text cleaning | NLTK | 3.8+ |
| Sentiment | VADER, TextBlob | Current |
| Topic modelling | Gensim LDA | 4.x |
| Visualisation | Matplotlib, Seaborn, Plotly, pyLDAvis | Current |
| Progress tracking | tqdm | Current |
| Python | 3.11+ | 3.11 |

---

## 📚 References

- Blei, D.M., Ng, A.Y., Jordan, M.I. (2003). *Latent Dirichlet Allocation*. JMLR.
- Hutto, C., Gilbert, E. (2014). *VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text*. ICWSM.
- Loria, S. (2018). *TextBlob Documentation*. textblob.readthedocs.io
- Řehůřek, R., Sojka, P. (2010). *Software Framework for Topic Modelling with Large Corpora*. ELRA.